In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('/content/Netflix_Cleaned.csv')

df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [8]:
# Step 1: Analyze rating categories

rating_missing = df['rating'].isna().sum()

rating_counts = df['rating'].value_counts()

rating_percentage = df['rating'].value_counts(normalize=True) * 100

rating_summary = pd.DataFrame({
    'Count': rating_counts,
    'Percentage': rating_percentage
})

print(f"Missing ratings: {rating_missing}")
print()
print(rating_summary)

Missing ratings: 0

          Count  Percentage
rating                     
TV-MA      3205   36.461889
TV-14      2157   24.539249
TV-PG       861    9.795222
R           799    9.089875
PG-13       490    5.574516
TV-Y7       333    3.788396
TV-Y        306    3.481229
PG          287    3.265074
TV-G        220    2.502844
NR           79    0.898749
G            41    0.466439
TV-Y7-FV      6    0.068259
NC-17         3    0.034130
UR            3    0.034130


In [9]:
# Step 2: Identify the most common genres

genre_data = df['listed_in'].dropna().str.split(', ').explode()

genre_counts = genre_data.value_counts()

genre_percentage = genre_counts / len(df) * 100

genre_summary = pd.DataFrame({
    'Count': genre_counts,
    'Percentage of Titles': genre_percentage
})

genre_summary.head(10)

,Count,Percentage of Titles
listed_in,,
International Movies,2752,31.308305
Dramas,2426,27.599545
Comedies,1674,19.044369
International TV Shows,1349,15.346985
Documentaries,869,9.886234
Action & Adventure,859,9.772469
TV Dramas,762,8.668942
Independent Movies,756,8.600683
Children & Family Movies,641,7.292378


In [10]:
# Step 3: Compare ratings across content types

rating_by_type = pd.crosstab(df['rating'], df['type'])

rating_by_type_percentage = pd.crosstab(
    df['rating'],
    df['type'],
    normalize='columns'
) * 100

print("Rating Counts by Content Type")
print(rating_by_type)

print("\nRating Percentages Within Each Content Type")
print(rating_by_type_percentage.round(2))

Rating Counts by Content Type
type      Movie  TV Show
rating                  
G            41        0
NC-17         3        0
NR           75        4
PG          287        0
PG-13       490        0
R           797        2
TV-14      1427      730
TV-G        126       94
TV-MA      2062     1143
TV-PG       540      321
TV-Y        131      175
TV-Y7       139      194
TV-Y7-FV      5        1
UR            3        0

Rating Percentages Within Each Content Type
type      Movie  TV Show
rating                  
G          0.67     0.00
NC-17      0.05     0.00
NR         1.22     0.15
PG         4.68     0.00
PG-13      8.00     0.00
R         13.01     0.08
TV-14     23.29    27.40
TV-G       2.06     3.53
TV-MA     33.66    42.91
TV-PG      8.81    12.05
TV-Y       2.14     6.57
TV-Y7      2.27     7.28
TV-Y7-FV   0.08     0.04
UR         0.05     0.00


In [11]:
# Step 4: Create interactive visualization

import plotly.express as px

rating_comparison = (
    rating_by_type_percentage
    .reset_index()
    .melt(
        id_vars='rating',
        var_name='Content Type',
        value_name='Percentage'
    )
)

fig = px.bar(
    rating_comparison,
    x='rating',
    y='Percentage',
    color='Content Type',
    barmode='group',
    title='Rating Distribution: Movies vs TV Shows',
    labels={
        'rating': 'Rating',
        'Percentage': 'Percentage (%)',
        'Content Type': 'Content Type'
    }
)

fig.update_layout(
    yaxis_title='Percentage within Content Type',
    xaxis_title='Rating',
    hovermode='x unified'
)

fig.show()

## Audience Insights

- TV-MA is the most common rating in the dataset, accounting for 36.46% of all titles.
- TV-14 is the second most common rating, accounting for 24.54% of all titles.
- TV Shows have a higher share of TV-MA content than Movies (42.91% vs 33.66%).
- TV Shows also have a higher share of TV-14 content than Movies (27.40% vs 23.29%).
- International Movies are the most common genre, appearing in 2,752 titles.
- Dramas are the second most common genre, appearing in 2,426 titles, followed by Comedies with 1,674 titles.
- Overall, the dataset shows a strong presence of mature-rated content, while International Movies, Dramas, and Comedies are among the most frequently represented genres.

In [12]:
# Additional interactive visualization: Top 10 Genres

top_genres = genre_counts.head(10).reset_index()
top_genres.columns = ['Genre', 'Count']

fig = px.bar(
    top_genres,
    x='Count',
    y='Genre',
    orientation='h',
    title='Top 10 Most Common Genres on Netflix',
    labels={
        'Count': 'Number of Titles',
        'Genre': 'Genre'
    }
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    hovermode='y'
)

fig.show()